In [2]:
# Define a custom enhanced FederatorRoutingClient
import obspy
from obspy.clients.fdsn.routing.federator_routing_client import FederatorRoutingClient
from obspy.clients.fdsn.header import FDSNNoDataException, FDSNException
from multiprocessing.dummy import Pool as ThreadPool
import traceback
import sys
import warnings
from urllib.parse import urlparse

def _try_download_bulk(r):
    try:
        return _download_bulk(r)
    except Exception:
        reason = "".join(traceback.format_exception(*sys.exc_info()))
        warnings.warn(
            "Failed to download data of type '%s' from '%s' due to: \n%s" % (
                r["data_type"], r["endpoint"], reason))
        return None


def _download_bulk(r):
    try:
        c = obspy.clients.fdsn.client.Client(r["endpoint"], debug=r["debug"],
                          timeout=r["timeout"])
    except FDSNException as e: 
        msg = e.args[0]
        msg += "It will not be used for routing. Try again later?"
        warnings.warn(msg)
        return None

    if r["data_type"] == "waveform":
        fct = c.get_waveforms_bulk
        service = c.services["dataselect"]
    elif r["data_type"] == "station":
        fct = c.get_stations_bulk
        service = c.services["station"]

    # Keep only kwargs that are supported by this particular service.
    kwargs = {k: v for k, v in r["kwargs"].items() if k in service}
    bulk_str = ""
    for key, value in kwargs.items():
        bulk_str += "%s=%s\n" % (key, str(value))
    try:
        return fct(bulk_str + r["bulk_str"])
    except FDSNException:
        return None


class EnhancedFederatorRoutingClient(FederatorRoutingClient):
    """
    Enhanced FederatorRoutingClient that allows more control over the request.
    """

    def __init__(
        self, debug=False, timeout=120, include_providers=None, exclude_providers=None
    ):
        """
        Initialize the client.

        Parameters
        ----------
        debug : bool, optional
            If True, print out debug information.
        timeout : int, optional
            Timeout in seconds.
        include_providers : list of str, optional
            List of providers to include. If not given, all providers are
            included.
        exclude_providers : list of str, optional
            List of providers to exclude. If not given, no providers are
            excluded.
        """
        super().__init__(
            url="http://service.iris.edu/irisws/fedcatalog/1",
            debug=debug,
            timeout=timeout,
            include_providers=include_providers,
            exclude_providers=exclude_providers,
        )

    def get_available_channels(self, **kwargs):
        """
        Get available channels from the federator.

        Accepted parameters are available from http://service.iris.edu/irisws/fedcatalog/1/.
        """
        # parameters that should be passed
        params = {k: str(kwargs[k]) for k in self.kwargs_of_interest if k in kwargs}
        params["format"] = "request" 

        # request the available channels
        r = self._download(self._url + "/query", params=params, content_type="text/plain")
        
        # split the responses for multiple datacenters
        records = self._split_routing_response(
            r.content.decode() if hasattr(r.content, "decode") else r.content,
            service="station")
        # filter requests based on including and excluding providers
        records = self._filter_requests(records)

        if not records:
            raise FDSNNoDataException(
                "Nothing remains to download after the provider "
                "inclusion/exclusion filters have been applied.")
        self.records = records
    
    def _download_waveforms(self, **kwargs):
        return self._download_parallel(self.records, data_type="waveform", **kwargs)

    def _download_stations(self, **kwargs):
        return self._download_parallel(self.records, data_type="station", **kwargs)
    
    def _download_parallel(self, split, data_type, **kwargs):
        # One thread per data center.
        dl_requests = []
        for k, v in split.items():
            dl_requests.append({
                "debug": self._debug,
                "timeout": self._timeout,
                "endpoint": k,
                "bulk_str": v,
                "data_type": data_type,
                "kwargs": kwargs,
            })
        pool = ThreadPool(processes=len(dl_requests))
        results = pool.map(_try_download_bulk, dl_requests)

        # Merge all results into a single object.
        if data_type == "waveform":
            collection = obspy.Stream()
        elif data_type == "station":
            collection = obspy.Inventory(
                networks=[],
                source="ObsPy FDSN Routing %s" % obspy.__version__)
        else:  # pragma: no cover
            raise ValueError

        for _i in results:
            if not _i:
                continue
            collection += _i

        # Explitly close the thread pool as somehow this does not work
        # automatically under linux. See #2342.
        pool.close()

        return collection


class Record:
    def __init__(self, net, sta, loc, cha, starttime, endtime):
        self.net = net
        self.sta = sta
        self.loc = loc
        self.cha = cha
        self.starttime = starttime
        self.endtime = endtime
        self.latitude = None
        self.longitude = None
    
    def __str__(self) -> str:
        return f"{self.net} {self.sta} {self.loc} {self.cha} {self.starttime} {self.endtime}"

def filter_records(records):
    inv = client._download_stations(records, level="channel")
    # convert the records to a dict of lists of Record objects
    for k, v in records.items():
        records[k] = [Record(*rec.split()) for rec in v.splitlines()]

    # maintain a dict of latitudes and longitudes
    latitudes, longitudes = {}, {}
    for net in inv:
        for sta in net:
            key = f"{net.code}.{sta.code}"
            latitudes[key] = sta.latitude
            longitudes[key] = sta.longitude

    # assign the latitudes and longitudes to the records
    for k, v in records.items():
        for rec in v:
            key = f"{rec.net}.{rec.sta}"
            if key not in latitudes.keys():
                continue
            rec.latitude = latitudes[key]
            rec.longitude = longitudes[key]

    # convert the records back to a dict of strings
    for k, v in records.items():
        records[k] = "\n".join([str(rec) for rec in v])

    return records


client = EnhancedFederatorRoutingClient(timeout=10)

client.get_available_channels(
    channel="BHZ",
    starttime="1995-11-14T06:32:55.750000Z",
    endtime="1995-11-14T06:35:55.750000Z",
)
client.records = filter_records(client.records)

inv = client._download_stations(level="response")
st = client._download_waveforms()
print(inv)
print(len(st))

/tmp/ipykernel_1437018/1185297303.py:29: UserWarning: No FDSN services could be discovered at 'http://www.orfeus-eu.org'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)
/tmp/ipykernel_1437018/1185297303.py:29: UserWarning: No FDSN services could be discovered at 'https://eida-sc3.infp.ro'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)
/tmp/ipykernel_1437018/1185297303.py:29: UserWarning: No FDSN services could be discovered at 'http://www.orfeus-eu.org'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)
/tmp/ipykernel_1437018/1185297303.py:29: UserWarning: No FDSN services could be discovered at 'https://eida-sc3.infp.ro'. This could be due to a temporary service outage or

Inventory created at 2023-09-13T08:21:52.717317Z
	Created by: ObsPy 1.4.0
		    https://www.obspy.org
	Sending institution: IRIS-DMC,ObsPy FDSN Routing 1.4.0,RESIF-SI,SeisComP,eXistDB (BGR,GFZ,INGV-ONT,IRIS-DMC,RESIF-DC,UIB,USP)
	Contains:
		Networks (45):
			AZ, BL (2x), BN, CD, CN, CZ (2x), EI, G, GE (2x), GR, GT, HU, IC, 
			II, IM, IS, IU, KN, KZ, LB, LD, MN, NL, NO (2x), NS, PS, PT, TW, UO
			US, UW, XA, XB, XC, XE, XG, XH, XI, XJ, XT, ZB
		Stations (409):
			AZ.BZN (Buzz Northerns Place, Anza, CA, USA)
			AZ.CRY (Cary Ranch, Anza, CA, USA)
			AZ.FRD (Ford Ranch, Anza, CA, USA)
			AZ.KNW (Keenwild Fire Station, Mountain Center, CA, USA)
			AZ.LVA2 (Lost Valley Scout Camp, CA, USA)
			AZ.PFO (Pinyon Flats Observatory, CA, USA)
			AZ.RDM (Red Mountain, Riverside Co, CA, USA)
			AZ.SND (Jim Saunders Place, Anza, CA, USA)
			AZ.WMC (Walmic Ranch, Anza, CA, USA)
			BL.JFOB (Juiz de Fora, MG)
			BL.TRRB (Tres Rios, RJ)
			BL.VABB (Estacao VABB, BLSP-Net)
			BN.EKB (ESKDALEMUIR)
			CD.EN

/tmp/ipykernel_1437018/1185297303.py:29: UserWarning: No FDSN services could be discovered at 'http://www.orfeus-eu.org'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)
/tmp/ipykernel_1437018/1185297303.py:29: UserWarning: No FDSN services could be discovered at 'https://eida-sc3.infp.ro'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)
